# Cameo Tag Update — Existing Extracted Model

A focused recipe for updating tags on requirements that have **already been extracted**
from a Cameo `.mdzip` model on the Istari platform.

Use this notebook when you have an existing model on the platform and want to write new
tag values back into it — without re-uploading the file or running a fresh extraction
first.

### What we cover

- Finding an already-uploaded Cameo model by ID or external identifier.
- Locating the most recent `@istari:extract` job and reading its `requirements.json` output.
- Browsing all requirements so you can pick the one to update.
- Running `@istari:update_tags` to write a tag back into the model.
- Optionally verifying the write with a fresh extraction.

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**.
- An agent with the **Cameo** integration and access to `@istari:update_tags`
  (and `@istari:extract` if you want to run the verification step).
- A Cameo model that has **already been uploaded and extracted** on the platform.
  If you need to upload first, use `cameo_extract_and_update_notebook - demo.ipynb`.

### Credentials

Create a `.env` file next to this notebook:

```
ISTARI_REGISTRY_URL=https://...your platform registry URL...
ISTARI_PERSONAL_ACCESS_TOKEN=...your token...
```


## 1 &middot; Connect and verify

In [ ]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition

platform = IstariPlatform.from_env()

report = platform.client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(platform)


## 2 &middot; Configure

Set **one** of `MODEL_ID` or `MODEL_EXTERNAL_ID` — whichever you have handy.
`MODEL_ID` takes precedence if both are set.

`TOOL_VERSION` and `OPERATING_SYSTEM` control which agent picks up the update
job; leave them empty to let the platform assign any compatible agent.


In [ ]:
# ── Identify the model ──────────────────────────────────────────────────────
MODEL_ID          = ""   # UUID from the platform UI or a previous notebook run
MODEL_EXTERNAL_ID = "cameo-requirements-extraction-tutorial-ncxtable-demo"  # set if no MODEL_ID

# ── Requirement to update ────────────────────────────────────────────────────
# Leave blank to browse all requirements first (Step 4), then come back and fill this in.
TARGET_REQUIREMENT_NAME = ""   # e.g. "Requirement REQ-001 Plate Thickness"

# ── Tag to write ─────────────────────────────────────────────────────────────
TAG_KEY   = "Text"              # tag name in the Cameo model
TAG_VALUE = "Selected part number: PN12345"  # value to write
REPLACE_EXISTING = False        # True = overwrite any existing value for this tag

# ── Agent targeting (optional) ───────────────────────────────────────────────
TOOL_VERSION      = ""   # e.g. "2024x-refresh2" — leave empty to auto-assign
OPERATING_SYSTEM  = ""   # e.g. "Windows 11"     — leave empty to auto-assign

# ── Verification (optional) ──────────────────────────────────────────────────
RUN_VERIFY = True   # set False to skip the re-extraction verification in Step 6


## 3 &middot; Find the existing model

Look up the model by `MODEL_ID` (direct, instant) or by `MODEL_EXTERNAL_ID`
(resource search, slightly slower but works when you only have the external
identifier you set at upload time).


In [ ]:
assert MODEL_ID or MODEL_EXTERNAL_ID, (
    "Set MODEL_ID or MODEL_EXTERNAL_ID in the config cell."
)

if MODEL_ID:
    model = platform.get_model(MODEL_ID)
    print(f"Fetched model by ID: {model.id}")
else:
    resource = (
        platform.resources()
        .type("model")
        .filter(external_identifier=[MODEL_EXTERNAL_ID])
        .sort("-created")
        .first()
    )
    assert resource is not None, (
        f"No model found with external_identifier={MODEL_EXTERNAL_ID!r}. "
        "Check the value or use MODEL_ID instead."
    )
    model = platform.get_model(resource.id)
    print(f"Found model by external_id: {model.id}")

print(f"  display_name : {model.name}")
print(f"  external_id  : {getattr(model._resource, 'external_identifier', None)}")


## 4 &middot; Find the latest extraction and browse requirements

We look at the jobs that have run on this model, find the most recent
`@istari:extract` that completed successfully, and read its `requirements.json`
output.

This lets you see every requirement — element IDs, names, and current tags —
so you can fill in `TARGET_REQUIREMENT_NAME` in the config cell above before
running Step 5.


In [ ]:
# Find the most recent completed @istari:extract job on this model
extract_job = None
for job in model.get_jobs():
    fn = getattr(job, "function_name", None) or ""
    status = str(getattr(job, "status", "")).upper()
    if "@istari:extract" in fn and "COMPLETED" in status:
        extract_job = job
        break   # get_jobs() returns newest-first

assert extract_job is not None, (
    "No completed @istari:extract job found for this model. "
    "Run an extraction first (see cameo_extract_and_update_notebook - demo.ipynb)."
)
print(f"Using extraction job: {extract_job.id}  ({extract_job.status})")

# Read requirements.json from that job
requirements_artifact = extract_job.find_product(filename="requirements.json")
assert requirements_artifact is not None, (
    "requirements.json not found in the latest extraction job. "
    f"Available products: {[p.name for p in extract_job.get_products()]}"
)

requirements_data = requirements_artifact.read_json()
print(f"\nFound {len(requirements_data)} requirement(s):\n")
for req in requirements_data:
    print(f"  name : {req['name']}")
    print(f"  id   : {req['id']}")
    print(f"  tags : {req.get('tags', {})}")
    print()


## 5 &middot; Pick the target requirement and run `@istari:update_tags`

After browsing the list above, fill in `TARGET_REQUIREMENT_NAME` (and the tag
values) in the config cell, then run this cell.

`@istari:update_tags` runs on the **original `.mdzip` model** and commits the
changes back as a new revision of that same model — not as a separate artifact.


In [ ]:
assert TARGET_REQUIREMENT_NAME, (
    "Set TARGET_REQUIREMENT_NAME in the config cell (Step 2) before running this."
)

# Find the target requirement by name
target_req = next(
    (r for r in requirements_data if r["name"] == TARGET_REQUIREMENT_NAME), None
)
assert target_req is not None, (
    f"Requirement {TARGET_REQUIREMENT_NAME!r} not found.\n"
    f"Available names:\n" + "\n".join(f"  - {r['name']}" for r in requirements_data)
)

TARGET_ELEMENT_ID = target_req["id"]
print(f"Target requirement:")
print(f"  name         : {target_req['name']}")
print(f"  element_id   : {TARGET_ELEMENT_ID}")
print(f"  current tags : {target_req.get('tags', {})}")
print(f"\nWriting  {TAG_KEY!r}: {TAG_VALUE!r}  (replace_existing={REPLACE_EXISTING})\n")

# Build the update_tags job
update_kwargs = {}
if TOOL_VERSION:
    update_kwargs["tool_version"] = TOOL_VERSION
if OPERATING_SYSTEM:
    update_kwargs["operating_system"] = OPERATING_SYSTEM

update_tags = JobDefinition(
    function="@istari:update_tags",
    tool_name="dassault_cameo",
    **update_kwargs,
    parameters={
        "updates": [
            {
                "element_id": TARGET_ELEMENT_ID,
                "replace_existing": REPLACE_EXISTING,
                "tags": {TAG_KEY: TAG_VALUE},
            }
        ]
    },
)

update_job = model.submit_job(update_tags)
print(f"Submitted tag-update job {update_job.id}; polling...")

update_job.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob finished: {update_job.status}")
products = update_job.get_products()
print(f"\nProducts ({len(products)}):")
for p in products:
    print(f"  - {p.type:10s}  name={p.name!r}  rev={p.revision_id}")


## 6 &middot; Verify the update (optional)

Run a fresh `@istari:extract` on the model and confirm the tag value now
appears in `requirements.json`.  Skip this cell (`RUN_VERIFY = False` in
the config) if you just want to trust the job succeeded.


In [ ]:
if not RUN_VERIFY:
    print("(verification skipped — set RUN_VERIFY = True in the config cell to enable)")
else:
    verify_extract = JobDefinition(
        function="@istari:extract",
        tool_name="dassault_cameo",
        **update_kwargs,
    )

    verify_job = model.submit_job(verify_extract)
    print(f"Submitted verification extraction {verify_job.id}; polling...")

    verify_job.wait(
        timeout=600,
        on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
    ).on_success()

    print(f"\nVerification job finished: {verify_job.status}")

    updated_artifact = verify_job.find_product(filename="requirements.json")
    assert updated_artifact is not None

    updated_data = updated_artifact.read_json()
    updated_req = next(
        (r for r in updated_data if r["id"] == TARGET_ELEMENT_ID), None
    )
    assert updated_req is not None, "Target requirement not found in updated extraction"

    print(f"\nUpdated requirement:")
    print(f"  name : {updated_req['name']}")
    print(f"  tags : {updated_req.get('tags', {})}")

    expected_value = TAG_VALUE
    actual_value = updated_req.get("tags", {}).get(TAG_KEY)
    if actual_value == expected_value:
        print(f"\n✓  Tag '{TAG_KEY}' correctly set to {actual_value!r}")
    else:
        print(f"\n✗  Expected {expected_value!r}, got {actual_value!r}")


## 7 &middot; Trace the lineage

`get_lineage()` walks backward from any revision and shows every step that
produced it.  If you ran the verification step, tracing from
`updated_artifact` shows the full chain: original upload → extract → update
→ re-extract.  Otherwise we trace from the model produced by the update job.


In [ ]:
trace_from = updated_artifact if RUN_VERIFY else update_job.get_products()[0]
tree = trace_from.get_lineage(max_depth=10)
print(f"Lineage for {trace_from.name!r}:\n")
tree.print_tree()


## Verify in the UI

Sign into the platform and cross-check:

1. **Files / Models** — the model should show a new revision produced by the update job.
2. **Jobs / Activity** — you should see one `@istari:update_tags` job (and one `@istari:extract`
   if you ran the verification step).
3. **Resources** — on the update job, confirm the updated model revision was produced.
4. **Verification** — on the verification extraction job, open `requirements.json` and confirm
   your tag appears under the target requirement.
